In [31]:
import pandas as pd
import numpy as np
import pandahouse
import scipy.stats as stats

In [ ]:
# подключаемся к базе данных и выгружаем данные по пользователям 
# (user_id, их эксперементальную группу, лайки, просмотры и ctr) из 1 и 2 группы

connection = {
    'host': 'http://clickhouse.lab.karpov.courses:8123',
    'password': '',
    'user': 'student',
    'database': 'simulator_20261020'
}

q = """
 SELECT
    user_id,
    exp_group,
    SUM(action = 'like') AS likes,
    SUM(action = 'view') AS views,
    likes / views AS ctr
 FROM {db}.feed_actions
 GROUP BY user_id, exp_group
"""

users_by_group = pandahouse.read_clickhouse(q, connection=connection)

users_by_group

,user_id,exp_group,likes,views,ctr
0,143844,3,40,170,0.235294
1,124673,4,40,120,0.333333
2,138472,2,60,406,0.147783
3,146334,2,8,31,0.258065
4,38237,3,84,284,0.295775
...,...,...,...,...,...
122621,107746,2,110,504,0.218254
122622,164670,3,34,236,0.144068
122623,63551,1,10,53,0.188679
122624,13171,1,53,245,0.216327


In [33]:
# Пусть группы 0 и 1 — это наши контрольные группы. Чему равен в них общегрупповой CTR (в виде доли)? 
# Укажите значение до 2 знака после точки.

control_groups = users_by_group[users_by_group['exp_group'].isin([0, 1])]

ctr_control_groups = control_groups.groupby('exp_group', as_index=False).agg(likes = ('likes', 'sum'),
                                                                                 views = ('views', 'sum'))

ctr_group_0 = ctr_control_groups[ctr_control_groups.exp_group == 0].likes / ctr_control_groups[ctr_control_groups.exp_group == 0].views
ctr_group_1 = ctr_control_groups[ctr_control_groups.exp_group == 1].likes / ctr_control_groups[ctr_control_groups.exp_group == 1].views

print(f'ctr в группе 0: {ctr_group_0.item():.2f}\nctr в группе 1: {ctr_group_1.item():.2f}')

ctr в группе 0: 0.21
ctr в группе 1: 0.21


In [35]:
# подключаемся к базе данных и выгружаем данные по пользователям 
# (user_id, их эксперементальную группу, лайки, просмотры и ctr) из 0 и 3 группы

q_2 = """
 SELECT
    user_id,
    exp_group,
    SUM(action = 'like') AS likes,
    SUM(action = 'view') AS views,
    likes / views AS ctr
 FROM {db}.feed_actions
 WHERE exp_group IN (0, 3)
    AND toDate(time) BETWEEN '2026-10-02' AND '2026-10-08'
 GROUP BY user_id, exp_group 
"""

users_by_group_2 = pandahouse.read_clickhouse(q_2, connection=connection)

users_by_group_2

,user_id,exp_group,likes,views,ctr
0,115383,3,12,44,0.272727
1,123580,3,2,11,0.181818
2,4944,0,8,41,0.195122
3,4504,0,5,15,0.333333
4,121508,0,6,25,0.240000
...,...,...,...,...,...
19917,13781,3,10,27,0.370370
19918,130651,3,18,98,0.183673
19919,3681,0,7,55,0.127273
19920,122008,3,5,41,0.121951


In [40]:
# Начнём с групп 0 и 3! Давайте сравним их t-тестом — сначала возьмём обычный пользовательский CTR, 
# а затем линеаризованные лайки. Что у нас получилось?

control_group = users_by_group_2[users_by_group_2['exp_group'] == 0].copy()
test_group = users_by_group_2[users_by_group_2['exp_group'] == 3].copy()

K = control_group['likes'].sum() / control_group['views'].sum()

control_group['linearized_likes'] = control_group['likes'] - K * control_group['views']
test_group['linearized_likes'] = test_group['likes'] - K * test_group['views']

ctr_ttest = stats.ttest_ind(test_group['ctr'],
                            control_group['ctr'],
                            equal_var=False)
linearized_ttest = stats.ttest_ind(test_group['linearized_likes'],
                                   control_group['linearized_likes'],
                                   equal_var=False)

print(f'результаты t-test для 0 и 3 групп:\nctr - {ctr_ttest}\nlinearized_likes - {linearized_ttest}')

результаты t-test для 0 и 3 групп:
ctr - Ttest_indResult(statistic=13.935320516755773, pvalue=6.216047483062228e-44)
linearized_likes - Ttest_indResult(statistic=16.186230032932844, pvalue=1.4918137745326139e-58)


In [37]:
# подключаемся к базе данных и выгружаем данные по пользователям 
# (user_id, их эксперементальную группу, лайки, просмотры и ctr) из 1 и 2 группы

q_3= """
 SELECT
    user_id,
    exp_group,
    SUM(action = 'like') AS likes,
    SUM(action = 'view') AS views,
    likes / views AS ctr
 FROM {db}.feed_actions
 WHERE exp_group IN (1, 2)
    AND toDate(time) BETWEEN '2026-10-02' AND '2026-10-08'
 GROUP BY user_id, exp_group 
"""

users_by_group_3 = pandahouse.read_clickhouse(q_3, connection=connection)

users_by_group_3

,user_id,exp_group,likes,views,ctr
0,109963,1,3,15,0.200000
1,26117,1,32,141,0.226950
2,138232,1,18,73,0.246575
3,26295,1,39,141,0.276596
4,18392,1,7,32,0.218750
...,...,...,...,...,...
19892,17146,2,0,10,0.000000
19893,30872,1,11,59,0.186441
19894,25383,2,42,127,0.330709
19895,139020,2,17,41,0.414634


In [39]:
# То же самое для групп групп 1 и 2! Сравниваем t-тесты — сначала обычный пользовательский CTR, 
# а затем линеаризованные лайки

control_group_2 = users_by_group_3[users_by_group_3['exp_group'] == 1].copy()
test_group_2 = users_by_group_3[users_by_group_3['exp_group'] == 2].copy()

K = control_group_2['likes'].sum() / control_group_2['views'].sum()

control_group_2['linearized_likes'] = control_group_2['likes'] - K * control_group_2['views']
test_group_2['linearized_likes'] = test_group_2['likes'] - K * test_group_2['views']

ctr_ttest_2 = stats.ttest_ind(test_group_2['ctr'],
                            control_group_2['ctr'],
                            equal_var=False)
linearized_ttest_2 = stats.ttest_ind(test_group_2['linearized_likes'],
                                   control_group_2['linearized_likes'],
                                   equal_var=False)

print(f'результаты t-test для 1 и 2 групп:\nctr - {ctr_ttest_2}\nlinearized_likes - {linearized_ttest_2}')

результаты t-test для 1 и 2 групп:
ctr - Ttest_indResult(statistic=-0.4051491913112757, pvalue=0.685373331140751)
linearized_likes - Ttest_indResult(statistic=-5.936377101934482, pvalue=2.9805064038667734e-09)
